In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])

import torch
from google.colab import drive
from ultralytics import YOLO

In [ ]:
import os

drive.mount("/content/drive")
root = "/content/drive/MyDrive/HM_img"

for sub in ["train/images", "train/labels", "valid/images", "valid/labels", "test_imgs"]:
    os.makedirs(f"{root}/{sub}", exist_ok=True)

exts = (".jpg", ".jpeg", ".png")

for split in ["train", "valid"]:
    imgs = {os.path.splitext(f)[0] for f in os.listdir(f"{root}/{split}/images") if f.lower().endswith(exts)}
    lbls = {os.path.splitext(f)[0] for f in os.listdir(f"{root}/{split}/labels") if f.endswith(".txt")}
    print(f"{split}: 이미지 {len(imgs)}장 / 라벨 {len(lbls)}개")
    if imgs - lbls:
        print("  라벨 없는 이미지:", sorted(imgs - lbls)[:5])
    if lbls - imgs:
        print("  이미지 없는 라벨:", sorted(lbls - imgs)[:5])

n_test = len([f for f in os.listdir(f"{root}/test_imgs") if f.lower().endswith(exts)])
print("test_imgs:", n_test, "장")

In [ ]:
drive.mount("/content/drive")
root = "/content/drive/MyDrive/HM_img"
yaml_path = "/content/drive/MyDrive/HM_img/data.yaml"

yaml_text = """
path: /content/drive/MyDrive/HM_img
train:
  - train/images
val:
  - valid/images
test:
  - test/images
nc: 2
names:
  0: crow_house
  1: pole
"""

with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(yaml_text)

model = YOLO("yolov8n.pt")
model.train(
    data=yaml_path,
    epochs=100,
    patience=20,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project=f"{root}/runs",
    name="Hail_Mary"
)

In [ ]:
import glob
from PIL import Image
from IPython.display import display

best = YOLO(f"{root}/runs/Hail_Mary/weights/best.pt")

metrics = best.val()
print("mAP50:", metrics.box.map50, "mAP50-95:", metrics.box.map)

paths = sorted(
    p for p in glob.glob(f"{root}/test_imgs/*")
    if p.lower().endswith((".jpg", ".jpeg", ".png"))
)
print(len(paths), "장")

results = best(paths, conf=0.25, imgsz=640, save=True, project=f"{root}/runs", name="test_pred")

for p, r in zip(paths, results):
    print(p.split("/")[-1])
    display(Image.fromarray(r.plot()[..., ::-1]))
    for box in r.boxes:
        print(f"  {r.names[int(box.cls)]:<12} conf={float(box.conf):.2f}")